In [3]:
!pip install tensorflow transformers torch nltk pandas numpy scikit-learn plotly google-generativeai

In [8]:
!ls /content/

sample_data  train.txt


In [10]:
import pandas as pd

train_df = pd.read_csv(
    '/content/train.txt',
    sep=';',
    names=['text', 'emotion']
)

train_df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [11]:
X = train_df['text']
y = train_df['emotion']

print(X.head())
print(y.head())

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: object
0    sadness
1    sadness
2      anger
3       love
4      anger
Name: emotion, dtype: object


In [12]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print(y[:10])

[4 4 0 3 0 4 5 1 2 3]


In [13]:
print(encoder.classes_)

['anger' 'fear' 'joy' 'love' 'sadness' 'surprise']


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Create tokenizer
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X)

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(X)

# Pad sequences
X = pad_sequences(sequences, maxlen=100)

print(X.shape)

(16000, 100)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (12800, 100)
Testing data shape: (3200, 100)


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=10000, output_dim=128),

    Bidirectional(LSTM(64)),

    Dropout(0.5),

    Dense(64, activation='relu'),

    Dense(6, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.5225 - loss: 1.2244 - val_accuracy: 0.7169 - val_loss: 0.7326
Epoch 2/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.8560 - loss: 0.4164 - val_accuracy: 0.8856 - val_loss: 0.3397
Epoch 3/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9350 - loss: 0.1957 - val_accuracy: 0.9056 - val_loss: 0.2820
Epoch 4/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9615 - loss: 0.1166 - val_accuracy: 0.9062 - val_loss: 0.3491
Epoch 5/5
400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9687 - loss: 0.0985 - val_accuracy: 0.8706 - val_loss: 0.4553


In [18]:
model.save('bilstm_model.h5')

print("Model saved successfully!")

Model saved successfully!


In [19]:
import pickle

# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

print("Tokenizer and Label Encoder saved successfully!")

Tokenizer and Label Encoder saved successfully!


In [20]:
sample_text = ["I am very happy because I completed my project"]

sequence = tokenizer.texts_to_sequences(sample_text)
padded = pad_sequences(sequence, maxlen=100)

prediction = model.predict(padded)

predicted_class = prediction.argmax(axis=1)[0]

emotion = encoder.inverse_transform([predicted_class])

print("Predicted Emotion:", emotion[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 585ms/step
Predicted Emotion: joy


In [21]:
# Predict on test data
y_pred = model.predict(X_test)

# Convert probabilities to class labels
y_pred_classes = np.argmax(y_pred, axis=1)

print("Predictions completed successfully!")

100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Predictions completed successfully!


In [22]:
import plotly.express as px
import pandas as pd

# Convert predicted labels back to emotion names
emotion_labels = encoder.inverse_transform(y_pred_classes)

# Create DataFrame
results_df = pd.DataFrame({
    'Emotion': emotion_labels
})

# Count emotions
emotion_counts = results_df['Emotion'].value_counts().reset_index()
emotion_counts.columns = ['Emotion', 'Count']

# Create chart
fig = px.bar(
    emotion_counts,
    x='Emotion',
    y='Count',
    title='Emotion Distribution',
    color='Emotion'
)

fig.show()

In [23]:
from datetime import datetime

# Create logs DataFrame
logs_df = pd.DataFrame({
    'Predicted_Emotion': emotion_labels
})

# Add timestamp
logs_df['Timestamp'] = datetime.now()

# Save to CSV
logs_df.to_csv('emotion_logs.csv', index=False)

print("Emotion logs saved successfully!")

Emotion logs saved successfully!


In [24]:
def learning_support(emotion):

    if emotion == 'sadness':
        return "You seem sad. Take a short break and stay positive."

    elif emotion == 'anger':
        return "You seem frustrated. Try revising the topic slowly and relax."

    elif emotion == 'fear':
        return "Don't worry. Practice more and build confidence."

    elif emotion == 'joy':
        return "Great! You are feeling happy. Continue your learning journey."

    elif emotion == 'love':
        return "You are feeling positive. Keep learning with enthusiasm."

    elif emotion == 'surprise':
        return "You seem surprised. Explore more about this topic."

    elif emotion == 'confused':
        return "You seem confused. Revise the concepts and watch tutorial videos."

    elif emotion == 'frustrated':
        return "Take a short break, relax, and try solving simpler problems first."

    elif emotion == 'curious':
        return "Great curiosity! Explore advanced topics and experiment with new ideas."

    elif emotion == 'confident':
        return "Excellent! You are confident. Move on to more challenging topics."

    elif emotion == 'bored':
        return "Try interactive learning methods such as videos, quizzes, or projects."

    else:
        return "Stay motivated and continue learning."

In [25]:
emotion = "joy"

print("Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Emotion: joy
AI Guidance: Great! You are feeling happy. Continue your learning journey.


In [26]:
user_text = input("Enter how you feel: ")

# Convert text to sequence
sequence = tokenizer.texts_to_sequences([user_text])

# Pad sequence
padded = pad_sequences(sequence, maxlen=100)

# Predict emotion
prediction = model.predict(padded)

predicted_class = np.argmax(prediction)

emotion = encoder.inverse_transform([predicted_class])[0]

# Display results
print("\nPredicted Emotion:", emotion)
print("AI Guidance:", learning_support(emotion))

Enter how you feel: I am very happy because I completed my project
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

Predicted Emotion: joy
AI Guidance: Great! You are feeling happy. Continue your learning journey.


In [27]:
user_text = input("Enter how you feel: ")

# Convert text to sequence
sequence = tokenizer.texts_to_sequences([user_text])

# Pad sequence
padded = pad_sequences(sequence, maxlen=100)

# Predict emotion
prediction = model.predict(padded)

predicted_class = np.argmax(prediction)

emotion = encoder.inverse_transform([predicted_class])[0]

# Confidence Score
confidence = np.max(prediction) * 100

# Display results
print("\nPredicted Emotion:", emotion)
print("Confidence Score: {:.2f}%".format(confidence))
print("AI Guidance:", learning_support(emotion))

Enter how you feel: I am very happy because I completed my project
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

Predicted Emotion: joy
Confidence Score: 62.00%
AI Guidance: Great! You are feeling happy. Continue your learning journey.


In [28]:
# Find top 2 emotions
top_2 = prediction[0].argsort()[-2:][::-1]

primary_emotion = encoder.inverse_transform([top_2[0]])[0]
secondary_emotion = encoder.inverse_transform([top_2[1]])[0]

print("Primary Emotion:", primary_emotion)
print("Secondary Emotion:", secondary_emotion)

Primary Emotion: joy
Secondary Emotion: anger


In [29]:
!pip install -q google-generativeai

In [30]:
import google.generativeai as genai

print("Gemini library imported successfully!")

Gemini library imported successfully!


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning:



All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md




In [31]:
import google.generativeai as genai

# Configure Gemini API
genai.configure(api_key="YOUR_API_KEY")

# Load Gemini model
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini configured successfully!")

Gemini configured successfully!


In [33]:
import google.generativeai as genai

genai.configure(api_key="PASTE_YOUR_API_KEY_HERE")

gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini configured successfully!")

Gemini configured successfully!


In [36]:
import google.generativeai as genai

genai.configure(api_key="PASTE_YOUR_API_KEY_HERE")

gemini_model = genai.GenerativeModel("gemini-1.5-flash")

print("Gemini configured successfully!")

Gemini configured successfully!


In [37]:
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [38]:
# Load Gemini model
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

prompt = f"""
The student is feeling {emotion}.

Provide short motivational learning guidance in 2-3 lines.
"""

response = gemini_model.generate_content(prompt)

print("\nGemini AI Guidance:")
print(response.text)


Gemini AI Guidance:
Fantastic! Let this joyful feeling fuel your curiosity and propel you to explore even more exciting knowledge. Keep embracing the wonder of learning!


In [40]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.7 MB/s eta 0:00:00


In [43]:
%%writefile app.py

import streamlit as st
import pickle
import numpy as np
import google.generativeai as genai

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Page Configuration
st.set_page_config(
    page_title="Emotion Detection & Learning Support Engine",
    page_icon="😊",
    layout="centered"
)

# Load Model
model = load_model('bilstm_model.h5')

with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)

with open('label_encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

# Gemini API Configuration
genai.configure(api_key="AQ.Ab8RN6KCEwmj96kMrMF-GFwTl3htBHkXTl5ZBGlGCge0-LTlFA")

gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

# Streamlit UI
st.title("😊 Emotion Detection & Learning Support Engine")

st.write("""
This application detects emotions from user text and provides AI-powered learning guidance.
""")

user_text = st.text_area(
    "Enter how you feel:",
    placeholder="Example: I am very happy because I completed my project"
)

if st.button("Detect Emotion"):

    if user_text.strip() == "":
        st.warning("Please enter some text.")

    else:

        # Text Preprocessing
        sequence = tokenizer.texts_to_sequences([user_text])
        padded = pad_sequences(sequence, maxlen=100)

        # Prediction
        prediction = model.predict(padded)

        predicted_class = np.argmax(prediction)

        emotion = encoder.inverse_transform([predicted_class])[0]

        confidence = np.max(prediction) * 100

        # Mixed Emotion Detection
        top_2 = prediction[0].argsort()[-2:][::-1]

        primary_emotion = encoder.inverse_transform([top_2[0]])[0]
        secondary_emotion = encoder.inverse_transform([top_2[1]])[0]

        # Display Results
        st.success(f"Predicted Emotion: {emotion}")

        st.info(f"Confidence Score: {confidence:.2f}%")

        st.write(f"### Primary Emotion: {primary_emotion}")
        st.write(f"### Secondary Emotion: {secondary_emotion}")

        # Gemini Guidance
        try:
            prompt = f"""
            The student is feeling {emotion}.

            Provide short motivational learning guidance in 2-3 lines.
            """

            response = gemini_model.generate_content(prompt)

            st.subheader("🤖 Gemini AI Guidance")
            st.write(response.text)

        except Exception as e:
            st.error(f"Gemini Error: {e}")

Overwriting app.py


In [44]:
!pip install transformers torch

In [45]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

print("BERT model loaded successfully!")

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

BERT model loaded successfully!


In [46]:
text = "I am very happy because I completed my project"

result = classifier(text)

print(result)

[[{'label': 'joy', 'score': 0.9896017909049988}, {'label': 'surprise', 'score': 0.005325485952198505}, {'label': 'sadness', 'score': 0.0024701592046767473}, {'label': 'anger', 'score': 0.0009551590774208307}, {'label': 'neutral', 'score': 0.0009251231676898897}, {'label': 'fear', 'score': 0.000414560257922858}, {'label': 'disgust', 'score': 0.00030774844344705343}]]


## BERT-based Emotion Analysis

A transformer-based emotion classification model
(j-hartmann/emotion-english-distilroberta-base)
was integrated to compare performance with the BiLSTM model.

### Sample Prediction

Input:
"I am very happy because I completed my project"

Output:
- Joy: 98.96%
- Surprise: 0.53%
- Sadness: 0.25%

### Model Comparison

| Model | Description |
|--------|-------------|
| BiLSTM | Deep learning sequence model trained on emotion dataset |
| BERT (DistilRoBERTa) | Pre-trained transformer model used for comparative emotion analysis |